In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model=512, num_q_heads=8, num_kv_groups=2):
        super().__init__()
        self.d_model = d_model
        self.num_q_heads = num_q_heads
        self.num_kv_groups = num_kv_groups
        self.num_q_per_kv = num_q_heads // num_kv_groups # Number of Q heads per KV head group
        
        self.head_dim = d_model // num_q_heads
        
        # Projections
        self.q_proj = nn.Linear(d_model, num_q_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, num_kv_groups * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, num_kv_groups * self.head_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, kv_cache=None):
        batch_size, seq_len, _ = x.shape
        
        # 1. Project Query, Key, and Value
        # Q Shape: (Batch, Seq_Len, Num_Q_Heads, Head_Dim) -> Transpose to (Batch, Num_Q_Heads, Seq_Len, Head_Dim)
        q = self.q_proj(x).view(batch_size, seq_len, self.num_q_heads, self.head_dim).transpose(1, 2)
        
        # K, V Shapes: (Batch, Seq_Len, Num_KV_Groups, Head_Dim) -> Transpose to (Batch, Num_KV_Groups, Seq_Len, Head_Dim)
        k = self.k_proj(x).view(batch_size, seq_len, self.num_kv_groups, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.num_kv_groups, self.head_dim).transpose(1, 2)

        # 2. Update KV Cache if provided
        if kv_cache is not None:
            past_k, past_v = kv_cache
            k = torch.cat([past_k, k], dim=-2)
            v = torch.cat([past_v, v], dim=-2)
        new_kv_cache = (k, v)

        # 3. Repeat/Expand KV heads to match Query Head count per group
        # Expand KV heads from (Batch, Num_KV_Groups, Seq_Len, Head_Dim) -> (Batch, Num_Q_Heads, Seq_Len, Head_Dim)
        k_expanded = k.repeat_interleave(self.num_q_per_kv, dim=1)
        v_expanded = v.repeat_interleave(self.num_q_per_kv, dim=1)

        # 4. Compute Scaled Dot-Product Attention
        scores = torch.matmul(q, k_expanded.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        # Apply causal mask
        causal_mask = torch.tril(torch.ones(seq_len, k_expanded.size(-2), device=x.device))
        scores = scores.masked_fill(causal_mask == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_output = torch.matmul(attn_weights, v_expanded)

        # 5. Concatenate heads and project back to d_model
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.out_proj(attn_output), new_kv_cache

# --- Audit GQA Memory Footprint ---
gqa_layer = GroupedQueryAttention(d_model=512, num_q_heads=8, num_kv_groups=2)
mock_input = torch.randn(2, 10, 512) # Batch=2, Seq=10, Dim=512

output, cache = gqa_layer(mock_input)

print("--- Grouped-Query Attention Verification ---")
print(f"Output Matrix Shape:       {output.shape} (Batch, Seq_Len, d_model)")
print(f"Cached Key Tensor Shape:   {cache[0].shape} (Batch, Num_KV_Groups, Seq_Len, Head_Dim)")
print(f"Notice that KV heads = {cache[0].shape[1]}, while Query heads = 8. Memory is cut by 4x!")

--- Grouped-Query Attention Verification ---
Output Matrix Shape:       torch.Size([2, 10, 512]) (Batch, Seq_Len, d_model)
Cached Key Tensor Shape:   torch.Size([2, 2, 10, 64]) (Batch, Num_KV_Groups, Seq_Len, Head_Dim)
Notice that KV heads = 2, while Query heads = 8. Memory is cut by 4x!
